In [1]:
import pandas as pd

In [2]:
flights = pd.read_parquet(
    "../data/processed/flights.parquet"
)

weather = pd.read_parquet(
    "../data/processed/weather.parquet"
)

In [3]:
flights[
    ["origin_airport", "scheduled_departure"]
].dtypes

origin_airport                    str
scheduled_departure    datetime64[us]
dtype: object

In [4]:
weather[
    ["airport_icao", "datetime", "timezone"]
].dtypes

airport_icao               str
datetime        datetime64[us]
timezone                   str
dtype: object

In [5]:
print(f"Flights: {flights.shape}")
print(f"Weather: {weather.shape}")

Flights: (3505891, 29)
Weather: (7398504, 9)


In [6]:
airport_timezones = (
    weather[
        ["airport_icao", "timezone"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

airport_timezones.head()

,airport_icao,timezone
0,SBAC,America/Fortaleza
1,SBAE,America/Sao_Paulo
2,SBAQ,America/Sao_Paulo
3,SBAR,America/Maceio
4,SBAT,America/Cuiaba


In [7]:
airport_timezones.shape

(211, 2)

In [8]:
airport_timezones[
    "airport_icao"
].value_counts().head()

airport_icao
SBAC    1
SBAE    1
SBAQ    1
SBAR    1
SBAT    1
Name: count, dtype: int64

In [9]:
airport_timezones["timezone"].value_counts()

timezone
America/Sao_Paulo       103
America/Fortaleza        21
America/Manaus           15
America/Belem            12
America/Bahia            12
America/Cuiaba           11
America/Santarem         10
America/Campo_Grande      6
America/Recife            6
America/Porto_Velho       4
America/Araguaina         3
America/Maceio            2
America/Rio_Branco        2
America/Eirunepe          2
America/Boa_Vista         1
America/Noronha           1
Name: count, dtype: int64

In [10]:
flights = flights.merge(
    airport_timezones,
    left_on="origin_airport",
    right_on="airport_icao",
    how="left",
    validate="many_to_one",
)

In [11]:
print(flights.shape)

flights["timezone"].isna().sum()

(3505891, 31)


np.int64(0)

In [12]:
flights["scheduled_departure_local"] = pd.NaT

for timezone, index in flights.groupby("timezone").groups.items():
    local_time = (
        flights.loc[index, "scheduled_departure"]
        .dt.tz_localize("America/Sao_Paulo")
        .dt.tz_convert(timezone)
        .dt.tz_localize(None)
    )

    flights.loc[index, "scheduled_departure_local"] = local_time

In [13]:
flights["scheduled_departure_hour"] = (
    flights["scheduled_departure_local"]
    .dt.floor("h")
)

In [15]:
for airport in ["SBGR", "SBEG", "SBRB"]:
    display(
        flights.loc[
            flights["origin_airport"] == airport,
            [
                "origin_airport",
                "scheduled_departure",
                "timezone",
                "scheduled_departure_local",
                "scheduled_departure_hour",
            ],
        ].head(5)
    )

,origin_airport,scheduled_departure,timezone,scheduled_departure_local,scheduled_departure_hour
1,SBGR,2022-01-01 00:05:00,America/Sao_Paulo,2022-01-01 00:05:00,2022-01-01 00:00:00
2,SBGR,2022-01-01 22:30:00,America/Sao_Paulo,2022-01-01 22:30:00,2022-01-01 22:00:00
3,SBGR,2022-01-01 23:20:00,America/Sao_Paulo,2022-01-01 23:20:00,2022-01-01 23:00:00
4,SBGR,2022-01-01 21:50:00,America/Sao_Paulo,2022-01-01 21:50:00,2022-01-01 21:00:00
7,SBGR,2022-01-02 00:05:00,America/Sao_Paulo,2022-01-02 00:05:00,2022-01-02 00:00:00


,origin_airport,scheduled_departure,timezone,scheduled_departure_local,scheduled_departure_hour
398,SBEG,2022-01-03 07:00:00,America/Manaus,2022-01-03 06:00:00,2022-01-03 06:00:00
408,SBEG,2022-01-03 14:55:00,America/Manaus,2022-01-03 13:55:00,2022-01-03 13:00:00
420,SBEG,2022-01-03 18:30:00,America/Manaus,2022-01-03 17:30:00,2022-01-03 17:00:00
421,SBEG,2022-01-04 08:00:00,America/Manaus,2022-01-04 07:00:00,2022-01-04 07:00:00
423,SBEG,2022-01-04 12:35:00,America/Manaus,2022-01-04 11:35:00,2022-01-04 11:00:00


,origin_airport,scheduled_departure,timezone,scheduled_departure_local,scheduled_departure_hour
2449,SBRB,2022-01-01 13:45:00,America/Rio_Branco,2022-01-01 11:45:00,2022-01-01 11:00:00
8475,SBRB,2022-01-09 14:00:00,America/Rio_Branco,2022-01-09 12:00:00,2022-01-09 12:00:00
14208,SBRB,2022-01-16 14:00:00,America/Rio_Branco,2022-01-16 12:00:00,2022-01-16 12:00:00
18921,SBRB,2022-01-23 14:00:00,America/Rio_Branco,2022-01-23 12:00:00,2022-01-23 12:00:00
19570,SBRB,NaT,America/Rio_Branco,NaT,NaT


In [16]:
weather.duplicated(
    subset=["airport_icao", "datetime"]
).sum()

np.int64(0)

In [17]:
flight_weather = flights.merge(
    weather,
    left_on=[
        "origin_airport",
        "scheduled_departure_hour",
    ],
    right_on=[
        "airport_icao",
        "datetime",
    ],
    how="left",
    validate="many_to_one",
)

In [18]:
print(f"Flights antes: {len(flights):,}")
print(f"Após merge: {len(flight_weather):,}")

Flights antes: 3,505,891
Após merge: 3,505,891


In [19]:
flight_weather["precipitation"].isna().sum()

np.int64(100607)

In [20]:
flight_weather["precipitation"].isna().mean()

np.float64(0.02869655673835838)

In [21]:
flight_weather.loc[
    flight_weather["precipitation"].isna(),
    [
        "origin_airport",
        "scheduled_departure",
        "scheduled_departure_local",
        "scheduled_departure_hour",
    ],
].head(20)

,origin_airport,scheduled_departure,scheduled_departure_local,scheduled_departure_hour
11,SBGL,NaT,NaT,NaT
56,SBGL,NaT,NaT,NaT
76,SBGR,NaT,NaT,NaT
103,SBGL,NaT,NaT,NaT
122,SBGR,NaT,NaT,NaT
150,SBGL,NaT,NaT,NaT
164,SBGL,NaT,NaT,NaT
197,SBGL,NaT,NaT,NaT
204,SBSV,NaT,NaT,NaT
205,SBSV,NaT,NaT,NaT


In [22]:
missing_weather = flight_weather[
    flight_weather["precipitation"].isna()
]

missing_weather["scheduled_departure"].isna().value_counts()

scheduled_departure
True     100593
False        14
Name: count, dtype: int64

In [23]:
pd.crosstab(
    flight_weather["scheduled_departure"].isna(),
    flight_weather["precipitation"].isna(),
)

precipitation,False,True
scheduled_departure,,
False,3405284,14
True,0,100593


In [27]:
[col for col in flight_weather.columns if "timezone" in col]

['timezone_x', 'timezone_y']

In [26]:
missing_with_departure = flight_weather[
    flight_weather["precipitation"].isna()
    & flight_weather["scheduled_departure"].notna()
]

missing_with_departure[
    [
        "origin_airport",
        "scheduled_departure",
        "timezone_x",
        "scheduled_departure_local",
        "scheduled_departure_hour",
    ]
].sort_values("scheduled_departure")

,origin_airport,scheduled_departure,timezone_x,scheduled_departure_local,scheduled_departure_hour
3478306,SBGR,2026-01-01 00:05:00,America/Sao_Paulo,2026-01-01 00:05:00,2026-01-01 00:00:00
3505199,SBBE,2026-01-01 00:20:00,America/Belem,2026-01-01 00:20:00,2026-01-01 00:00:00
3505597,SBFL,2026-01-01 00:30:00,America/Sao_Paulo,2026-01-01 00:30:00,2026-01-01 00:00:00
3476796,SBBE,2026-01-01 00:35:00,America/Belem,2026-01-01 00:35:00,2026-01-01 00:00:00
3454050,SBGR,2026-01-01 01:30:00,America/Sao_Paulo,2026-01-01 01:30:00,2026-01-01 01:00:00
3478939,SBRF,2026-01-01 01:45:00,America/Recife,2026-01-01 01:45:00,2026-01-01 01:00:00
3505703,SBGL,2026-01-01 03:45:00,America/Sao_Paulo,2026-01-01 03:45:00,2026-01-01 03:00:00
3454051,SBGR,2026-01-01 04:00:00,America/Sao_Paulo,2026-01-01 04:00:00,2026-01-01 04:00:00
3505321,SBGR,2026-01-01 04:10:00,America/Sao_Paulo,2026-01-01 04:10:00,2026-01-01 04:00:00
3505322,SBGR,2026-01-01 05:45:00,America/Sao_Paulo,2026-01-01 05:45:00,2026-01-01 05:00:00


In [30]:
(flight_weather["timezone_x"] == flight_weather["timezone_y"]).value_counts(dropna=False)

True     3405284
False     100607
Name: count, dtype: int64

In [31]:
flights["scheduled_departure"].agg(["min", "max"])

min   2022-01-01 00:05:00
max   2026-01-02 09:05:00
Name: scheduled_departure, dtype: datetime64[us]

In [32]:
flights[
    flights["scheduled_departure"].dt.year > 2025
]["scheduled_departure"].dt.year.value_counts()

scheduled_departure
2026    14
Name: count, dtype: int64

In [33]:
flight_weather = flight_weather[
    flight_weather["scheduled_departure"].between(
        "2022-01-01",
        "2025-12-31 23:59:59"
    )
].copy()

In [34]:
flight_weather = flight_weather[
    flight_weather["scheduled_departure"].notna()
].copy()

In [36]:
flight_weather.shape

(3405284, 42)

In [37]:
flight_weather["precipitation"].isna().sum()

np.int64(0)

In [38]:
(flight_weather["timezone_x"]== flight_weather["timezone_y"]).value_counts(dropna=False)

True    3405284
Name: count, dtype: int64

In [39]:
flight_weather = flight_weather.drop(
    columns=[
        "airport_icao_y",
        "timezone_y",
    ],
    errors="ignore",
)

flight_weather = flight_weather.rename(
    columns={
        "airport_icao_x": "airport_icao",
        "timezone_x": "timezone",
    }
)

In [41]:
flight_weather.columns.tolist()

['airline_icao',
 'airline',
 'flight_number',
 'Código DI',
 'Código Tipo Linha',
 'aircraft_model',
 'number_of_seats',
 'origin_airport',
 'origin_airport_name',
 'scheduled_departure',
 'actual_departure',
 'destination_airport',
 'destination_airport_name',
 'scheduled_arrival',
 'actual_arrival',
 'flight_status',
 'Justificativa',
 'Referência',
 'Situação Partida',
 'Situação Chegada',
 'Codeshare',
 'departure_delay_minutes',
 'arrival_delay_minutes',
 'airport_icao',
 'airport_name',
 'city',
 'state',
 'latitude',
 'longitude',
 'timezone',
 'scheduled_departure_local',
 'scheduled_departure_hour',
 'datetime',
 'precipitation',
 'temperature_2m',
 'relative_humidity_2m',
 'wind_speed_10m',
 'wind_gusts_10m',
 'weather_code',
 'airport_icao']

In [42]:
flight_weather.shape

(3405284, 40)

In [44]:
flight_weather.columns[flight_weather.columns.duplicated()].tolist()

['airport_icao']

In [45]:
flight_weather = flight_weather.loc[
    :, ~flight_weather.columns.duplicated()
].copy()

In [46]:
flight_weather.columns.is_unique

True

In [47]:
flight_weather.shape

(3405284, 39)

In [48]:
flight_weather.to_parquet(
    "../data/processed/flight_weather.parquet",
    index=False,
)